In [12]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_diabetes
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from scipy import stats


In [3]:
from pathlib import Path
import pandas as pd
import tarfile
import urllib.request

def load_housing_data():
    tarball_path = Path("datasets/housing.tgz")
    if not tarball_path.is_file():
        Path("datasets").mkdir(parents=True, exist_ok=True)
        url = "https://github.com/ageron/data/raw/main/housing.tgz"
        urllib.request.urlretrieve(url, tarball_path)
        with tarfile.open(tarball_path) as housing_tarball:
            housing_tarball.extractall(path="datasets")
    return pd.read_csv(Path("datasets/housing/housing.csv"))

housing = load_housing_data()

In [4]:


X = housing.drop("median_house_value", axis=1)
y = housing["median_house_value"]


print(X.head())
print(y.head())
print(X.shape)
print(X.dtypes)


   longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
0    -122.23     37.88                41.0        880.0           129.0   
1    -122.22     37.86                21.0       7099.0          1106.0   
2    -122.24     37.85                52.0       1467.0           190.0   
3    -122.25     37.85                52.0       1274.0           235.0   
4    -122.25     37.85                52.0       1627.0           280.0   

   population  households  median_income ocean_proximity  
0       322.0       126.0         8.3252        NEAR BAY  
1      2401.0      1138.0         8.3014        NEAR BAY  
2       496.0       177.0         7.2574        NEAR BAY  
3       558.0       219.0         5.6431        NEAR BAY  
4       565.0       259.0         3.8462        NEAR BAY  
0    452600.0
1    358500.0
2    352100.0
3    341300.0
4    342200.0
Name: median_house_value, dtype: float64
(20640, 9)
longitude             float64
latitude              float64
housing_media

In [5]:
X.fillna(0)

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,NEAR BAY
...,...,...,...,...,...,...,...,...,...
20635,-121.09,39.48,25.0,1665.0,374.0,845.0,330.0,1.5603,INLAND
20636,-121.21,39.49,18.0,697.0,150.0,356.0,114.0,2.5568,INLAND
20637,-121.22,39.43,17.0,2254.0,485.0,1007.0,433.0,1.7000,INLAND
20638,-121.32,39.43,18.0,1860.0,409.0,741.0,349.0,1.8672,INLAND


In [7]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

X = pd.get_dummies(X.fillna(0))
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

initial_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Initial RMSE: {initial_rmse}")

# Cross-validation
scores = cross_val_score(model, X, y, scoring='neg_mean_squared_error', cv=5)

# Calculate the root mean squared error (RMSE) for each fold
rmse_scores = np.sqrt(-scores)

# Output the results
print("Root Mean Squared Error (RMSE) scores for each fold:", rmse_scores)
print("Mean RMSE:", rmse_scores.mean())
print("Standard Deviation of RMSE:", rmse_scores.std())

Initial RMSE: 69921.47177399474
Root Mean Squared Error (RMSE) scores for each fold: [71246.9881192  69190.35622159 68333.21235468 67787.07860332
 67872.1406122 ]
Mean RMSE: 68885.95518219852
Standard Deviation of RMSE: 1281.2502389224685


In [30]:
housing = load_housing_data()

In [31]:
X = housing.drop("median_house_value", axis=1)
y = housing["median_house_value"]

In [35]:
# Remove Outliers
Q1 = X.quantile(0.25)
Q3 = X.quantile(0.75)
IQR = Q3 - Q1

# Determine a mask for rows without outliers
mask = ~((X < (Q1 - 1.5 * IQR)) | (X > (Q3 + 1.5 * IQR))).any(axis=1)
X_clean = X[mask]
y_clean = y[mask]

print(f"Shape before cleaning: {X.shape}")
print(f"Shape after cleaning: {X_clean.shape}")


Shape before cleaning: (20640, 9)
Shape after cleaning: (18184, 9)


<ipython-input-35-5611ac1947ed>:1: FutureWarning: The default value of numeric_only in DataFrame.quantile is deprecated. In a future version, it will default to False. Select only valid columns or specify the value of numeric_only to silence this warning.
  Q1 = X.quantile(0.25)
<ipython-input-35-5611ac1947ed>:2: FutureWarning: The default value of numeric_only in DataFrame.quantile is deprecated. In a future version, it will default to False. Select only valid columns or specify the value of numeric_only to silence this warning.
  Q3 = X.quantile(0.75)
<ipython-input-35-5611ac1947ed>:6: FutureWarning: Automatic reindexing on DataFrame vs Series comparisons is deprecated and will raise ValueError in a future version. Do `left, right = left.align(right, axis=1, copy=False)` before e.g. `left == right`
  mask = ~((X < (Q1 - 1.5 * IQR)) | (X > (Q3 + 1.5 * IQR))).any(axis=1)


In [36]:
# Numeric attributes pipeline
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy="median")),
    ('scaler', StandardScaler()),
])

# Categorical attributes pipeline
cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))])

# Combine numerical and categorical pipelines
preprocessor = ColumnTransformer([
    ("num", num_pipeline, ["longitude", "latitude", "housing_median_age", "total_rooms",
                           "total_bedrooms", "population", "households", "median_income"]),
    ("cat", cat_pipeline, ["ocean_proximity"]),
])

X_clean = preprocessor.fit_transform(X_clean)


In [39]:
# Train on cleaned data
X_train_clean, X_test_clean, y_train_clean, y_test_clean = train_test_split(X_clean, y_clean, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train_clean, y_train_clean)
y_pred = model.predict(X_test_clean)

cleaned_rmse = np.sqrt(mean_squared_error(y_test_clean, y_pred))
print(f"Cleaned Data RMSE: {cleaned_rmse}")

# Cross-validation
scores = cross_val_score(model, X_clean, y_clean, scoring='neg_mean_squared_error', cv=5)

# Calculate the root mean squared error (RMSE) for each fold
rmse_scores = np.sqrt(-scores)

# Output the results
print("Root Mean Squared Error (RMSE) scores for each fold:", rmse_scores)
print("Mean RMSE:", rmse_scores.mean())
print("Standard Deviation of RMSE:", rmse_scores.std())

Cleaned Data RMSE: 64675.25896891283
Root Mean Squared Error (RMSE) scores for each fold: [68171.94724284 66980.70362458 66706.41880433 64041.35936847
 65534.57297469]
Mean RMSE: 66287.00040298208
Standard Deviation of RMSE: 1401.3715592135943
